<a href="https://colab.research.google.com/github/Tejnu/CSET-419---Generative-artificial-intelligence/blob/main/lab09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("output", exist_ok=True)

In [ ]:
dataset_text = """
machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
"""

sentences = [s.strip() for s in dataset_text.split("\n") if s.strip()]

In [ ]:
with open("output/dataset.txt","w") as f:
    f.write(dataset_text)

In [ ]:
tokens = []

for s in sentences:
    tokens.extend(s.split())

vocab = sorted(set(tokens))
word2idx = {w:i+1 for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

vocab_size = len(word2idx) + 1

with open("output/vocab.txt","w") as f:
    for w in vocab:
        f.write(w+"\n")

In [ ]:
SEQ_LEN = 3

X = []
Y = []

for s in sentences:
    words = s.split()

    for i in range(len(words)-SEQ_LEN):
        seq = words[i:i+SEQ_LEN]
        target = words[i+SEQ_LEN]

        X.append([word2idx[w] for w in seq])
        Y.append(word2idx[target])

In [ ]:
class SeqDataset(Dataset):

    def __init__(self,X,Y):
        self.X=torch.tensor(X)
        self.Y=torch.tensor(Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,i):
        return self.X[i],self.Y[i]

dataset = SeqDataset(X,Y)
loader = DataLoader(dataset,batch_size=16,shuffle=True)

In [ ]:
class LSTMGenerator(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()

        self.embed = nn.Embedding(vocab_size,64)
        self.lstm = nn.LSTM(64,128,batch_first=True)
        self.fc = nn.Linear(128,vocab_size)

    def forward(self,x):

        x = self.embed(x)
        out,_ = self.lstm(x)

        out = out[:,-1,:]
        out = self.fc(out)

        return out

In [ ]:
model = LSTMGenerator(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

EPOCHS = 500

loss_history=[]

for epoch in range(EPOCHS):

    total_loss = 0

    for x,y in loader:

        x=x.to(device)
        y=y.to(device)

        pred = model(x)

        loss = criterion(pred,y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(loader)

    print(f"Epoch {epoch+1} Loss {avg_loss:.4f}")

    loss_history.append(avg_loss)

Epoch 1 Loss 4.4964
Epoch 2 Loss 4.4098
Epoch 3 Loss 4.3356
Epoch 4 Loss 4.2602
Epoch 5 Loss 4.1808
Epoch 6 Loss 4.0961
Epoch 7 Loss 4.0002
Epoch 8 Loss 3.8942
Epoch 9 Loss 3.7719
Epoch 10 Loss 3.6339
Epoch 11 Loss 3.4722
Epoch 12 Loss 3.2842
Epoch 13 Loss 3.0744
Epoch 14 Loss 2.8360
Epoch 15 Loss 2.5667
Epoch 16 Loss 2.3011
Epoch 17 Loss 2.0274
Epoch 18 Loss 1.7495
Epoch 19 Loss 1.4882
Epoch 20 Loss 1.2433
Epoch 21 Loss 1.0254
Epoch 22 Loss 0.8331
Epoch 23 Loss 0.6813
Epoch 24 Loss 0.5564
Epoch 25 Loss 0.4575
Epoch 26 Loss 0.3756
Epoch 27 Loss 0.3158
Epoch 28 Loss 0.2681
Epoch 29 Loss 0.2309
Epoch 30 Loss 0.2019
Epoch 31 Loss 0.1773
Epoch 32 Loss 0.1585
Epoch 33 Loss 0.1424
Epoch 34 Loss 0.1290
Epoch 35 Loss 0.1175
Epoch 36 Loss 0.1080
Epoch 37 Loss 0.0996
Epoch 38 Loss 0.0920
Epoch 39 Loss 0.0857
Epoch 40 Loss 0.0797
Epoch 41 Loss 0.0749
Epoch 42 Loss 0.0703
Epoch 43 Loss 0.0662
Epoch 44 Loss 0.0624
Epoch 45 Loss 0.0591
Epoch 46 Loss 0.0561
Epoch 47 Loss 0.0533
Epoch 48 Loss 0.0507
E

In [ ]:
torch.save(model.state_dict(),"output/lstm_model.pt")

In [ ]:
with open("output/lstm_training_log.txt","w") as f:
    for i,l in enumerate(loss_history):
        f.write(f"Epoch {i+1} Loss {l}\n")

In [ ]:
def generate(model,seed,length=10):

    model.eval()

    words = seed.split()

    for _ in range(length):

        seq = words[-SEQ_LEN:]
        seq = [word2idx.get(w,0) for w in seq]

        x = torch.tensor([seq]).to(device)

        with torch.no_grad():
            pred = model(x)

        prob = torch.softmax(pred, dim=-1)

        idx = torch.multinomial(prob,1).item()

        next_word = idx2word.get(idx,"")

        words.append(next_word)

    return " ".join(words)

In [ ]:
samples=[]

seed = "machine learning models"

for i in range(5):

    g = generate(model,seed,8)
    print(g)
    samples.append(g)

machine learning models learn patterns from data. step. data. flow. applications.
machine learning models learn patterns from data. data. audio sequences. hidden
machine learning models learn patterns from data. data. for sequential tasks.
machine learning models learn patterns from data. data. step. designed for
machine learning models learn patterns from data. data. time steps. widely


In [ ]:
with open("output/lstm_generated_sequences.txt","w") as f:
    for s in samples:
        f.write(s+"\n")

In [ ]:
class TransformerGenerator(nn.Module):

    def __init__(self,vocab_size):

        super().__init__()

        self.embed = nn.Embedding(vocab_size,64)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.fc = nn.Linear(64,vocab_size)

    def forward(self,x):

        x = self.embed(x)

        x = x.permute(1,0,2)

        out = self.transformer(x)

        out = out[-1]

        out = self.fc(out)

        return out

In [ ]:
model_t = TransformerGenerator(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_t.parameters(),lr=0.001)

loss_history_t=[]

for epoch in range(EPOCHS):

    total_loss=0

    for x,y in loader:

        x=x.to(device)
        y=y.to(device)

        pred = model_t(x)

        loss = criterion(pred,y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()

    avg_loss = total_loss/len(loader)

    print(f"Epoch {epoch+1} Loss {avg_loss:.4f}")

    loss_history_t.append(avg_loss)

Epoch 1 Loss 4.7637
Epoch 2 Loss 3.4926
Epoch 3 Loss 2.7953


/tmp/ipykernel_6488/1766255300.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(


Epoch 4 Loss 2.1758
Epoch 5 Loss 1.8649
Epoch 6 Loss 1.5769
Epoch 7 Loss 1.3359
Epoch 8 Loss 1.1929
Epoch 9 Loss 1.0283
Epoch 10 Loss 0.9024
Epoch 11 Loss 0.8047
Epoch 12 Loss 0.7184
Epoch 13 Loss 0.6648
Epoch 14 Loss 0.5914
Epoch 15 Loss 0.5220
Epoch 16 Loss 0.4688
Epoch 17 Loss 0.4368
Epoch 18 Loss 0.3892
Epoch 19 Loss 0.3733
Epoch 20 Loss 0.3276
Epoch 21 Loss 0.3059
Epoch 22 Loss 0.2877
Epoch 23 Loss 0.2690
Epoch 24 Loss 0.2490
Epoch 25 Loss 0.2304
Epoch 26 Loss 0.2210
Epoch 27 Loss 0.2024
Epoch 28 Loss 0.1886
Epoch 29 Loss 0.1846
Epoch 30 Loss 0.1739
Epoch 31 Loss 0.1573
Epoch 32 Loss 0.1542
Epoch 33 Loss 0.1456
Epoch 34 Loss 0.1393
Epoch 35 Loss 0.1307
Epoch 36 Loss 0.1228
Epoch 37 Loss 0.1200
Epoch 38 Loss 0.1126
Epoch 39 Loss 0.1064
Epoch 40 Loss 0.1031
Epoch 41 Loss 0.1016
Epoch 42 Loss 0.0965
Epoch 43 Loss 0.0906
Epoch 44 Loss 0.0861
Epoch 45 Loss 0.0883
Epoch 46 Loss 0.0798
Epoch 47 Loss 0.0797
Epoch 48 Loss 0.0768
Epoch 49 Loss 0.0781
Epoch 50 Loss 0.0734
Epoch 51 Loss 0.068

In [ ]:
torch.save(model_t.state_dict(),"output/transformer_model.pt")

In [ ]:
samples_t=[]

for i in range(5):

    g = generate(model_t,"sequence models process",8)

    print(g)

    samples_t.append(g)

sequence models process data step by step. learn probability distributions. data.
sequence models process data step by step. step. from data. data.
sequence models process data step by step. steps. to training data.
sequence models process data step by step. hidden states across time
sequence models process data step by step. hidden states across time


In [ ]:
with open("output/transformer_generated_sequences.txt","w") as f:
    for s in samples_t:
        f.write(s+"\n")

In [ ]:
with open("output/evaluation.txt","w") as f:

    f.write("Sequence Generation Lab\n\n")

    f.write("Model 1 : LSTM\n")
    f.write("Model 2 : Transformer\n\n")

    f.write("Generated samples saved.\n")

In [ ]:
zip_path="lab9_output.zip"

with zipfile.ZipFile(zip_path,"w") as zipf:

    for root,dirs,files in os.walk("output"):
        for file in files:

            path=os.path.join(root,file)
            zipf.write(path)

print("Zip created:",zip_path)

Zip created: lab9_output.zip


In [ ]:
from google.colab import files
files.download("lab9_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>